# Feature Engineering

Извлекаю признаки из датасета при помощи XGBoost регрессии


In [3]:
import pandas as pd
import numpy as np
import librosa
from pathlib import Path
from tqdm.auto import tqdm

base_path = Path("..")
processed_path = base_path / "data" / "processed"
split_path = processed_path / "data_split"
train_path = split_path / "train.csv"
test_path = split_path / "test.csv"
val_path = split_path / "val.csv"


c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
val_df = pd.read_csv(val_path)
train_df.shape, test_df.shape, val_df.shape

((2415, 10), (520, 10), (515, 10))

#### Проверяем на 1

In [8]:
first_path = Path(train_df["wet_path"].iloc[0])
y, sr = librosa.load(first_path, sr=None)
type(y), y.shape, sr

(numpy.ndarray, (240000,), 48000)

In [9]:
def aggregate_feature(feature, f_name):

    f_mean = feature.mean()
    f_std = feature.std()
    f_median = np.median(feature)

    return {
        f"{f_name}_mean": f_mean,
        f"{f_name}_std": f_std,
        f"{f_name}_median": f_median,
    }

In [ ]:
rms = librosa.feature.rms(y=y)

aggregate_feature(rms, "rms")

(np.float32(0.02440065), np.float32(0.03990087), np.float32(0.006486482))

In [20]:
centroid = librosa.feature.spectral_centroid(y=y, sr=sr)

aggregate_feature(centroid, "centroid")

{'centroid_mean': np.float64(1799.3247257307717),
 'centroid_std': np.float64(728.5355114131153),
 'centroid_median': np.float64(1518.8766700121475)}

In [ ]:
bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)

aggregate_feature(bandwidth, "bandwidth")

(1, 469)


(np.float64(2517.4204691062873),
 np.float64(1288.787349766983),
 np.float64(2268.6445590713383))

In [19]:
rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)

aggregate_feature(rolloff, "rolloff")

{'rolloff_mean': np.float64(3022.2381396588485),
 'rolloff_std': np.float64(1843.799653170514),
 'rolloff_median': np.float64(2484.375)}

In [18]:
zrc = librosa.feature.zero_crossing_rate(y=y)

aggregate_feature(zrc, "zcr")

{'zcr_mean': np.float64(0.033993328558102345),
 'zcr_std': np.float64(0.011499821590852515),
 'zcr_median': np.float64(0.033203125)}

In [21]:
features = {}

features.update(aggregate_feature(rms, "rms"))
features.update(aggregate_feature(centroid, "centroid"))
features.update(aggregate_feature(bandwidth, "bandwidth"))
features.update(aggregate_feature(rolloff, "rolloff"))
features.update(aggregate_feature(zrc, "zcr"))

features

{'rms_mean': np.float32(0.02440065),
 'rms_std': np.float32(0.03990087),
 'rms_median': np.float32(0.006486482),
 'centroid_mean': np.float64(1799.3247257307717),
 'centroid_std': np.float64(728.5355114131153),
 'centroid_median': np.float64(1518.8766700121475),
 'bandwidth_mean': np.float64(2517.4204691062873),
 'bandwidth_std': np.float64(1288.787349766983),
 'bandwidth_median': np.float64(2268.6445590713383),
 'rolloff_mean': np.float64(3022.2381396588485),
 'rolloff_std': np.float64(1843.799653170514),
 'rolloff_median': np.float64(2484.375),
 'zcr_mean': np.float64(0.033993328558102345),
 'zcr_std': np.float64(0.011499821590852515),
 'zcr_median': np.float64(0.033203125)}

In [27]:
mfcc = librosa.feature.mfcc(
    y=y,
    sr=sr,
    n_mfcc=13
)
mfcc_features = {}

for i in range (mfcc.shape[0]):
    mfcc_features.update(aggregate_feature(mfcc[i], f"mfcc_{i+1}"))

print(list(mfcc_features.items())[:6])
len(mfcc_features)

[('mfcc_1_mean', np.float32(-483.36322)), ('mfcc_1_std', np.float32(102.59901)), ('mfcc_1_median', np.float32(-493.36777)), ('mfcc_2_mean', np.float32(137.37407)), ('mfcc_2_std', np.float32(62.755207)), ('mfcc_2_median', np.float32(142.5387))]


39

In [28]:
features.update(mfcc_features)

len(features)

54

#### Большая функция для всех признаков

In [33]:
def extract_features(audio_path):
    y, sr = librosa.load(audio_path, sr=None)

    features = {}

    rms = librosa.feature.rms(y=y)
    features.update(
        aggregate_feature(rms, "rms")
    )

    centroid = librosa.feature.spectral_centroid(
        y=y,
        sr=sr
    )
    features.update(
        aggregate_feature(centroid, "centroid")
    )

    bandwidth = librosa.feature.spectral_bandwidth(
        y=y,
        sr=sr
    )
    features.update(
        aggregate_feature(bandwidth, "bandwidth")
    )

    rolloff = librosa.feature.spectral_rolloff(
        y=y,
        sr=sr
    )
    features.update(
        aggregate_feature(rolloff, "rolloff")
    )

    zcr = librosa.feature.zero_crossing_rate(y=y)
    features.update(
        aggregate_feature(zcr, "zcr")
    )

    mfcc = librosa.feature.mfcc(
        y=y,
        sr=sr,
        n_mfcc=13
    )

    for i in range (mfcc.shape[0]):
        features.update(
            aggregate_feature(mfcc[i], f"mfcc_{i+1}")
        )


    return features

In [34]:
test_features = extract_features(first_path)

len(test_features)

54

In [35]:
for i in range(3):
    audio_path = Path(train_df["wet_path"].iloc[i])
    result = extract_features(audio_path)

    print(i, len(result))

0 54
1 54
2 54


#### Функция выявления признаков для всего датасета

In [44]:
def build_feature_dataframe(df):
    rows = []

    for _, row in tqdm(
        df.iterrows(),
        total=len(df)
    ):
        audio_path = Path(row["wet_path"])
        features = extract_features(audio_path)

        features["source_id"] = row["source_id"]
        features["wet_path"] = row["wet_path"]

        features["room_size_norm"] = row["room_size_norm"]
        features["wet_level_norm"] = row["wet_level_norm"]
        features["rate_hz_norm"] = row["rate_hz_norm"]
        features["depth_norm"] = row["depth_norm"]

        rows.append(features)

    return pd.DataFrame(rows)

In [49]:
small_df = build_feature_dataframe(train_df.head(3))

small_df.shape


100%|██████████| 3/3 [00:00<00:00,  8.08it/s]


(3, 60)

In [43]:
small_df[
    [
        "source_id",
        "wet_path",
        "room_size_norm",
        "wet_level_norm",
        "rate_hz_norm",
        "depth_norm",
    ]
]

,source_id,wet_path,room_size_norm,wet_level_norm,rate_hz_norm,depth_norm
0,Bridge_1-0,..\data\processed\wet\Bridge_1-0_v0.wav,0.374540,0.950714,0.731994,0.598658
1,Bridge_1-0,..\data\processed\wet\Bridge_1-0_v1.wav,0.156019,0.155995,0.058084,0.866176
2,Bridge_1-0,..\data\processed\wet\Bridge_1-0_v2.wav,0.601115,0.708073,0.020584,0.969910


#### Применяю ко всему train

In [50]:
train_features_df = build_feature_dataframe(train_df)
print(train_features_df.isna().sum().sum())
train_features_df.shape

100%|██████████| 2415/2415 [06:11<00:00,  6.50it/s]


0


(2415, 60)

In [51]:
val_features_df = build_feature_dataframe(val_df)
test_features_df = build_feature_dataframe(test_df)

print("Train:", train_features_df.shape)
print("Val:", val_features_df.shape)
print("Test:", test_features_df.shape)

print("Train NaN:", train_features_df.isna().sum().sum())
print("Val NaN:", val_features_df.isna().sum().sum())
print("Test NaN:", test_features_df.isna().sum().sum())

100%|██████████| 520/520 [01:33<00:00,  5.56it/s]


Train: (2415, 60)
Val: (515, 60)
Test: (520, 60)
Train NaN: 0
Val NaN: 0
Test NaN: 0


In [53]:
features_path = processed_path / "features"
features_path.mkdir(parents=True, exist_ok=True)

In [54]:
train_features_df.to_csv(
    features_path / "train_features.csv",
    index=False
)

val_features_df.to_csv(
    features_path / "val_features.csv",
    index=False
)

test_features_df.to_csv(
    features_path / "test_features.csv",
    index=False
)

In [55]:
list(features_path.iterdir())

[WindowsPath('../data/processed/features/test_features.csv'),
 WindowsPath('../data/processed/features/train_features.csv'),
 WindowsPath('../data/processed/features/val_features.csv')]

In [56]:
check_df = pd.read_csv(features_path / "train_features.csv")

check_df.shape

(2415, 60)

## Feature engineering 2
попробовал другие параметры и признаки

In [12]:
def extract_features_v2(audio_path):
    y, sr = librosa.load(audio_path, sr=None)

    features = {}


    rms = librosa.feature.rms(y=y)
    features.update(
        aggregate_feature(rms, "rms")
    )

    centroid = librosa.feature.spectral_centroid(
        y=y,
        sr=sr
    )
    features.update(
        aggregate_feature(centroid, "centroid")
    )

    bandwidth = librosa.feature.spectral_bandwidth(
        y=y,
        sr=sr
    )
    features.update(
        aggregate_feature(bandwidth, "bandwidth")
    )

    rolloff = librosa.feature.spectral_rolloff(
        y=y,
        sr=sr
    )
    features.update(
        aggregate_feature(rolloff, "rolloff")
    )

    zcr = librosa.feature.zero_crossing_rate(y=y)
    features.update(
        aggregate_feature(zcr, "zcr")
    )

    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    for i in range(chroma.shape[0]):
        features.update(
            aggregate_feature(chroma[i], f"chroma_{i+1}")
        )

    contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
    for i in range(contrast.shape[0]):
        features.update(
            aggregate_feature(contrast[i], f"contrast_{i+1}")
        )


    mfcc = librosa.feature.mfcc(
        y=y,
        sr=sr,
        n_mfcc=13
    )

    mfcc_delta = librosa.feature.delta(mfcc)
    
    mfcc_delta2 = librosa.feature.delta(
        mfcc,
        order=2
    )

    for i in range (mfcc.shape[0]):
        features.update(
            aggregate_feature(mfcc[i], f"mfcc_{i+1}")
        )

    for i in range(mfcc_delta.shape[0]):
        features.update(
            aggregate_feature(
                mfcc_delta[i],
                f"mfcc_delta_{i+1}"
            )
        )

    for i in range(mfcc_delta2.shape[0]):
        features.update(
            aggregate_feature(
                mfcc_delta2[i],
                f"mfcc_delta2_{i+1}"
            )
        )


    return features

In [14]:
first_path = Path(train_df.iloc[0]["wet_path"])

test_features_v2 = extract_features_v2(first_path)

len(test_features_v2), np.isnan(list(test_features_v2.values())).sum()

(189, np.int64(0))

In [15]:
def build_feature_dataframe_v2(df):
    rows = []

    for _, row in tqdm(df.iterrows(), total=len(df)):
        audio_path = Path(row["wet_path"])

        features = extract_features_v2(audio_path)

        features["source_id"] = row["source_id"]
        features["wet_path"] = row["wet_path"]

        features["room_size_norm"] = row["room_size_norm"]
        features["wet_level_norm"] = row["wet_level_norm"]
        features["rate_hz_norm"] = row["rate_hz_norm"]
        features["depth_norm"] = row["depth_norm"]

        rows.append(features)

    return pd.DataFrame(rows)

In [17]:
small_train_v2 = build_feature_dataframe_v2(
    train_df.head(5)
)

small_train_v2.shape, small_train_v2.isna().sum().sum()

100%|██████████| 5/5 [00:00<00:00,  5.16it/s]


((5, 195), np.int64(0))

In [18]:
train_features_v2 = build_feature_dataframe_v2(train_df)
val_features_v2 = build_feature_dataframe_v2(val_df)
test_features_v2 = build_feature_dataframe_v2(test_df)
print(train_features_v2.shape)
print(val_features_v2.shape)
test_features_v2.shape

100%|██████████| 520/520 [01:57<00:00,  4.43it/s]

(2415, 195)
(515, 195)


(520, 195)

In [19]:
features_v2_path = processed_path / "features_v2"

features_v2_path.mkdir(
    parents=True,
    exist_ok=True
)

train_features_v2.to_csv(
    features_v2_path / "train_features_v2.csv",
    index=False
)

val_features_v2.to_csv(
    features_v2_path / "val_features_v2.csv",
    index=False
)

test_features_v2.to_csv(
    features_v2_path / "test_features_v2.csv",
    index=False
)

In [20]:
list(features_v2_path.iterdir())

[WindowsPath('../data/processed/features_v2/test_features_v2.csv'),
 WindowsPath('../data/processed/features_v2/train_features_v2.csv'),
 WindowsPath('../data/processed/features_v2/val_features_v2.csv')]